# 02 — Deep Exploratory Data Analysis (EDA)

**Step 2.1** — Online Retail Analytics Platform | Dev B

This notebook performs a comprehensive business analysis on the cleaned dataset.
All analysis functions live in `src/analysis.py` so they can be reused directly in Streamlit.

### Analysis Sections
1. Overall KPI Summary
2. Revenue Trends (monthly, daily, by hour, by day-of-week)
3. Product Performance (top sellers, worst performers, return rates)
4. Customer Analysis (CLV, churn, new vs returning)
5. Geographic Performance
6. Return & Refund Analysis


## Setup

In [1]:
import pandas as pd
import numpy as np
import sys, os
sys.path.append(os.path.abspath('..'))

from src.data_cleaning import clean_data
from src.analysis import (
    get_kpi_summary,
    get_monthly_revenue, get_daily_revenue,
    get_revenue_by_hour, get_revenue_by_day_of_week,
    get_top_products, get_worst_products, get_product_return_rate,
    get_country_performance, get_customer_lifetime_value,
    get_churned_customers, get_new_vs_returning_customers,
    get_return_summary
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)


## Data Loading & Cleaning

In [2]:
import os
DATA_PATH = '../data/cleaned_retail_data.csv'
if not os.path.exists(DATA_PATH):
    print(f"File not found at {DATA_PATH}. Please run 01_data_exploration.ipynb first.")
else:
    print("Loading cleaned dataset...")
    df_clean = pd.read_csv(DATA_PATH, parse_dates=['InvoiceDate'])
    print(f"Cleaned shape: {df_clean.shape}")
    display(df_clean.head())

Loading cleaned dataset...


Cleaned shape: (534130, 10)


,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34


---
## 1. Overall KPI Summary

> **Business value:** Quick health-check of the entire business in one glance.


In [3]:
kpis = get_kpi_summary(df_clean)
for k, v in kpis.items():
    label = k.replace('_', ' ').title()
    print(f"  {label:<30} {v}")


  Total Revenue                  9748149.07
  Total Orders                   23796
  Avg Order Value                409.65
  Unique Customers               4371
  Top Country                    United Kingdom
  Best Product                   DOTCOM POSTAGE
  Return Rate %                  1.73


---
## 2. Revenue Trends

### 2a. Monthly Revenue + Growth Rate
> **Business value:** Identify best/worst months, seasonality patterns, and growth momentum.


In [4]:
monthly = get_monthly_revenue(df_clean)
monthly


,YearMonth,Revenue,Orders,Customers,Revenue_Growth_%
0,2010-12,"746,723.61",1885,949,NaN
1,2011-01,"558,448.56",1346,784,-25.21
2,2011-02,"497,026.41",1319,799,-11.00
3,2011-03,"682,013.98",1772,1021,37.22
4,2011-04,"492,367.84",1486,900,-27.81
5,2011-05,"722,094.10",1995,1080,46.66
6,2011-06,"689,977.23",1862,1052,-4.45
7,2011-07,"680,156.99",1745,994,-1.42
8,2011-08,"703,510.58",1639,981,3.43
9,2011-09,"1,017,596.68",2170,1303,44.65


### 2b. Revenue by Day of Week
> **Business value:** Know which days drive the most sales — useful for scheduling promotions.


In [5]:
dow = get_revenue_by_day_of_week(df_clean)
dow


,DayOfWeek,Revenue,Transactions
0,Monday,"1,584,895.30","94,080.00"
1,Tuesday,"1,965,703.61","100,468.00"
2,Wednesday,"1,730,088.43","93,195.00"
3,Thursday,"2,108,701.53","102,561.00"
4,Friday,"1,560,100.74","81,025.00"
5,Saturday,NaN,NaN
6,Sunday,"798,659.46","62,801.00"


### 2c. Revenue by Hour of Day
> **Business value:** Identify peak shopping hours — target email campaigns at the right time.


In [6]:
hourly = get_revenue_by_hour(df_clean)
hourly


,Hour,Revenue,Transactions
0,6,-497.35,41
1,7,"31,009.32",383
2,8,"281,723.02",8899
3,9,"766,524.17",34208
4,10,"1,327,329.89",48602
5,11,"1,146,457.49",56608
6,12,"1,357,613.12",77323
7,13,"1,172,985.87",71052
8,14,"1,113,532.86",66065
9,15,"1,186,819.41",76558


---
## 3. Product Performance

### 3a. Top 10 Best-Selling Products
> **Business value:** Focus marketing budget and stock replenishment on high-revenue products.


In [7]:
top_products = get_top_products(df_clean, n=10)
top_products


,Description,Revenue,Quantity,Transactions
1074,DOTCOM POSTAGE,"206,245.48",705,707
2864,REGENCY CAKESTAND 3 TIER,"164,459.49",12996,2187
3859,WHITE HANGING HEART T-LIGHT HOLDER,"99,612.42",35294,2353
2422,PARTY BUNTING,"98,243.88",18006,1719
1825,JUMBO BAG RED RETROSPOT,"92,175.79",47256,2153
2752,RABBIT NIGHT LIGHT,"66,661.63",30631,1032
2703,POSTAGE,"66,248.64",3004,1253
2390,PAPER CHAIN KIT 50'S CHRISTMAS,"63,715.24",18876,1194
228,ASSORTED COLOUR BIRD ORNAMENT,"58,792.42",36282,1488
751,CHILLI LIGHTS,"53,746.66",10222,673


### 3b. Bottom 10 Worst-Selling Products
> **Business value:** Products with very low revenue may be candidates for discontinuation or repricing.


In [8]:
worst_products = get_worst_products(df_clean, n=10)
worst_products


,Description,Revenue,Quantity
2352,PADS TO MATCH ALL CUSHIONS,0.00,3
1690,HEN HOUSE W CHICK IN NEST,0.42,1
3075,SET 12 COLOURING PENCILS DOILEY,0.65,1
3838,WHITE BEADED GARLAND STRING 20LIGHT,0.80,0
3711,VINTAGE BLUE TINSEL REEL,0.84,2
2527,PINK CRYSTAL GUITAR PHONE CHARM,0.85,1
1650,HAPPY BIRTHDAY CARD TEDDY/CAKE,0.95,5
667,CAT WITH SUNGLASSES BLANK CARD,0.95,5
62,3 WICK CHRISTMAS BRIAR CANDLE,0.97,-1
111,60 GOLD AND SILVER FAIRY CAKE CASES,1.10,2


### 3c. Product Return Rates
> **Business value:** High return rates signal quality issues, misleading descriptions, or wrong pricing.


In [9]:
return_rate = get_product_return_rate(df_clean)
return_rate.head(15)


,Description,Total,Returns,ReturnRate_%
0,FLAMINGO LIGHTS,1,1.00,100.00
1,BLUE FLYING SINGING CANARY,1,1.00,100.00
2,Discount,77,77.00,100.00
3,WHITE CHERRY LIGHTS,1,1.00,100.00
4,PINK LARGE JEWELED PHOTOFRAME,1,1.00,100.00
5,SWEETHEART KEY CABINET,1,1.00,100.00
6,TEA TIME CAKE STAND IN GIFT BOX,1,1.00,100.00
7,CRUK Commission,16,16.00,100.00
8,CREAM SWEETHEART TRAYS,1,1.00,100.00
9,CREAM SWEETHEART SHELF + HOOKS,1,1.00,100.00


---
## 4. Customer Analysis

### 4a. Customer Lifetime Value (CLV)
> **Business value:** Identify your most valuable customers for VIP treatment, loyalty rewards, or targeted upselling.


In [10]:
clv = get_customer_lifetime_value(df_clean)
print(f"Total customers: {len(clv)}")
print(f"\nTop 10 highest-value customers:")
clv.head(10)


Total customers: 4371

Top 10 highest-value customers:


,CustomerID,TotalRevenue,Orders,AvgOrderValue
1702,14646.0,"279,489.02",76,"3,677.49"
4232,18102.0,"256,438.49",62,"4,136.10"
3757,17450.0,"187,322.17",55,"3,405.86"
1894,14911.0,"132,458.73",248,534.11
55,12415.0,"123,725.45",26,"4,758.67"
1344,14156.0,"113,214.59",66,"1,715.37"
3800,17511.0,"88,125.38",46,"1,915.77"
3201,16684.0,"65,892.08",31,"2,125.55"
1004,13694.0,"62,690.54",60,"1,044.84"
2191,15311.0,"59,284.19",118,502.41


### 4b. CLV Distribution
> **Business value:** Understand what % of revenue comes from your top customers (Pareto effect).


In [11]:
top_pct = clv.head(int(len(clv) * 0.1))
top_revenue = top_pct['TotalRevenue'].sum()
total_revenue = clv['TotalRevenue'].sum()
print(f"Top 10% of customers generate {top_revenue/total_revenue*100:.1f}% of total revenue")

print("\nRevenue distribution quantiles:")
print(clv['TotalRevenue'].describe(percentiles=[.25, .5, .75, .90, .95]))


Top 10% of customers generate 60.1% of total revenue

Revenue distribution quantiles:
count     4,371.00
mean      1,893.97
std       8,219.59
min      -4,287.63
25%         291.94
50%         644.24
75%       1,608.94
90%       3,497.14
95%       5,601.45
max     279,489.02
Name: TotalRevenue, dtype: float64


### 4c. At-Risk / Churned Customers (90+ days inactive)
> **Business value:** Re-engage customers before they are permanently lost — cheaper than acquiring new ones.


In [12]:
churned = get_churned_customers(df_clean, days_threshold=90)
print(f"Customers inactive for 90+ days: {len(churned)}")
churned.head(10)


Customers inactive for 90+ days: 1454


,CustomerID,LastPurchaseDate,DaysSinceLastPurchase
3128,16583.0,2010-12-01 12:03:00,373
4095,17908.0,2010-12-01 11:45:00,373
4211,18074.0,2010-12-01 09:53:00,373
359,12791.0,2010-12-01 11:27:00,373
1045,13747.0,2010-12-01 10:37:00,373
4139,17968.0,2010-12-01 12:23:00,373
1763,14729.0,2010-12-01 12:43:00,373
406,12855.0,2010-12-02 09:37:00,372
4050,17855.0,2010-12-02 09:44:00,372
3969,17732.0,2010-12-02 09:29:00,372


### 4d. New vs Returning Customers per Month
> **Business value:** Track whether marketing efforts are bringing new customers, and whether existing ones stay loyal.


In [13]:
new_vs_ret = get_new_vs_returning_customers(df_clean)
new_vs_ret


,YearMonth,CustomerType,Customers
0,2010-12,New,948
1,2011-01,New,421
2,2011-01,Returning,362
3,2011-02,New,380
4,2011-02,Returning,418
5,2011-03,New,440
6,2011-03,Returning,580
7,2011-04,New,299
8,2011-04,Returning,600
9,2011-05,New,279


---
## 5. Geographic Performance
> **Business value:** Know which countries drive revenue — focus international marketing there, or investigate underperforming markets.


In [14]:
geo = get_country_performance(df_clean)
geo.head(15)


,Country,Revenue,UniqueCustomers,Orders
36,United Kingdom,"8,189,252.30",3950,21391
24,Netherlands,"284,661.54",9,100
10,EIRE,"262,993.38",4,360
14,Germany,"221,509.47",95,603
13,France,"197,335.11",88,461
0,Australia,"137,009.77",9,69
33,Switzerland,"56,363.05",22,74
31,Spain,"54,756.03",31,105
3,Belgium,"40,910.96",25,119
32,Sweden,"36,585.41",8,46


---
## 6. Return & Refund Analysis
> **Business value:** Returns reduce net revenue and increase operational costs. Knowing the scale helps prioritize quality improvements.


In [15]:
return_summary = get_return_summary(df_clean)
for k, v in return_summary.items():
    label = k.replace('_', ' ').title()
    print(f"  {label:<35} {v:,.2f}" if isinstance(v, float) else f"  {label:<35} {v:,}")


  Total Transactions                  534,130
  Return Transactions                 9,251
  Return Rate %                       1.73
  Revenue Lost From Returns           893,979.73
  Net Revenue                         9,748,149.07


---
## Key Business Findings

Results from running all cells above (dataset: 534,130 transactions, 4,371 customers).

| # | Finding | Recommended Action |
|---|---------|-------------------|
| 1 | Top 10% customers generate **60.1%** of revenue | Launch VIP loyalty program |
| 2 | Peak shopping hour is **12:00** (highest revenue & transactions) | Schedule email campaigns at this time |
| 3 | **1,454** customers inactive 90+ days | Send win-back discount campaign |
| 4 | Product "**FLAMINGO LIGHTS**" has highest return rate (100%) | Investigate quality / description |
| 5 | **Netherlands** is the top revenue country excl. UK (284,661 GBP) | Increase marketing investment there |

---
*Next step -> Step 2.2: RFM Segmentation + Recommendation Engine (`src/recommendation_engine.py`)*
